## 0. INTRODUCTION

Proyek ini difokuskan untuk mengembangkan model Custom CNN yang mampu mengklasifikasikan gambar lesi kulit secara multi-kelas terhadap kategori-kategori penyakit kulit tertentu dengan harapan model tersebut dapat berguna sebagai langkah awal dalam menciptakan sebuah sistem pendeteksi dini penyakit kulit.

Oleh:
- Achmad Baihaqie Wibowo (103012400026)
- Bayu Alif Aryo Wiputra (103012)

Catatan:
- `notebook_01_EDA.ipynb` difokuskan sebagai wadah untuk melaksanakan tahap eksplorasi dan analisis terhadap dataset.
- `notebook_02_DEVELOPMENT.ipynb` difokuskan sebagai wadah untuk melaksanakan tahap *data preparation* dan pengembangan model.
- tautan dataset: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000

## 1. Import Library and Initial Setup

In [1]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torchvision.transforms as transforms

# sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

# setup CUDA
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# setup reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
set_seed(42)

Using device: cuda


## 2. Load Data

In [2]:
# define data directory
BASE_DIR = 'data'

# define paths of dataset
metadata_path = os.path.join(BASE_DIR, 'HAM10000_metadata.csv')

image_folders = [
    os.path.join(BASE_DIR, 'HAM10000_images_part_1'), 
    os.path.join(BASE_DIR, 'HAM10000_images_part_2')
]

# define dictionary to store images' path
image_id_path_dict = {os.path.splitext(os.path.basename(x))[0]: x for folder in image_folders for x in glob(os.path.join(folder, "*.jpg"))}

# load metadata
df_metadata = pd.read_csv(metadata_path)

# load images' path by mapping image_ids into their true image_path
df_metadata['image_path'] = df_metadata['image_id'].map(image_id_path_dict)

## 3. Data Splitting

In [3]:
df_unique_lesions = df_metadata.drop_duplicates(subset='lesion_id')
ids_train_val, ids_test = train_test_split(df_unique_lesions['lesion_id'], test_size=0.10, stratify=df_unique_lesions['dx'], random_state=42)
df_unique_train_val = df_unique_lesions[df_unique_lesions['lesion_id'].isin(ids_train_val)]
ids_train, ids_val = train_test_split(df_unique_train_val['lesion_id'], test_size=10/90, stratify=df_unique_train_val['dx'], random_state=42)
df_train = df_metadata[df_metadata['lesion_id'].isin(ids_train)].copy().reset_index(drop=True)
df_val = df_metadata[df_metadata['lesion_id'].isin(ids_val)].copy().reset_index(drop=True)
df_test = df_metadata[df_metadata['lesion_id'].isin(ids_test)].copy().reset_index(drop=True)

In [4]:
# check for result of data splitting
print(f"train set: {len(df_train)} | val set: {len(df_val)} | test set: {len(df_test)}")
overlap_train_val = set(df_train['lesion_id']).intersection(set(df_val['lesion_id']))
print(f"jumlah ID bocor antara train set dan validation set: {len(overlap_train_val)}")
overlap_train_test = set(df_train['lesion_id']).intersection(set(df_test['lesion_id']))
print(f"jumlah ID bocor antara train set dan test set: {len(overlap_train_test)}")

# imbalanced class proportion
print("\nproporsi tiap kelas dalam train set:")
print(df_train['dx'].value_counts())

train set: 7979 | val set: 1012 | test set: 1024
jumlah ID bocor antara train set dan validation set: 0
jumlah ID bocor antara train set dan test set: 0

proporsi tiap kelas dalam train set:
dx
nv       5337
mel       885
bkl       884
bcc       412
akiec     258
vasc      114
df         89
Name: count, dtype: int64


*Inference*: Setelah *data splitting* dapat dipastikan bahwa tidak ada lesion_id yang sama berada di dua *set* yang berbeda (*train set* dengan [*test set* dan *validation set*]) sehingga dapat dipastikan sejauh ini *data leakage* berhasil ditangani.

## 4. Pre-processing and Feature Engineering

In [5]:
# handle missing values
df_train.loc[df_train['age'] < 0, 'age'] = np.nan
train_age_median_mapping = df_train.groupby('dx')['age'].median()
for df in [df_train, df_val, df_test]:
    df['age'] = df['age'].fillna(df['dx'].map(train_age_median_mapping))

In [6]:
# feature scaling: standardization
scaler = StandardScaler()
df_train['age_scaled'] = scaler.fit_transform(df_train[['age']])
df_val['age_scaled'] = scaler.transform(df_val[['age']])
df_test['age_scaled'] = scaler.transform(df_test[['age']])

In [7]:
# categorical feature encoding: manual label encoding
manual_map = {
    'akiec': 0, # Actinic keratoses
    'bcc':   1, # Basal cell carcinoma
    'bkl':   2, # Benign keratosis-like lesions
    'df':    3, # Dermatofibroma
    'mel':   4, # Melanoma
    'nv':    5, # Melanocytic nevi
    'vasc':  6  # Vascular lesions
}

for df in [df_train, df_val, df_test]:
    df['label_idx'] = df['dx'].map(manual_map).astype(int)
print(f"tipe data target label_idx: {df_train['label_idx'].dtype}")

tipe data target label_idx: int64


In [8]:
# categorical feature encoding: one hot encoding
ohe_columns = ['sex', 'localization']
df_train = pd.get_dummies(df_train, columns=ohe_columns)
df_val = pd.get_dummies(df_val, columns=ohe_columns)
df_test = pd.get_dummies(df_test, columns=ohe_columns)

# ensure all splits have the same dummy columns (align columns)
for df in [df_train, df_val, df_test]:
    for col in df_train.columns:
        if col not in df.columns:
            df[col] = 0
            
# keep order
df_train = df_train[df_train.columns]
df_val = df_val[df_train.columns]
df_test = df_test[df_train.columns]

In [9]:
# feature selection: contextual based
drop_cols = ['lesion_id', 'image_id', 'dx', 'dx_type', 'age']
for df in [df_train, df_val, df_test]:
    cols_to_drop = [c for c in drop_cols if c in df.columns]
    df.drop(columns=cols_to_drop, inplace=True)

In [12]:
# define list of metadata features
metadata_features = [col for col in df_train.columns if col not in ['label_idx', 'image_path']]
print(f"metadata features: {metadata_features}")
n_metadata = len(metadata_features)

metadata features: ['age_scaled', 'sex_female', 'sex_male', 'sex_unknown', 'localization_abdomen', 'localization_acral', 'localization_back', 'localization_chest', 'localization_ear', 'localization_face', 'localization_foot', 'localization_genital', 'localization_hand', 'localization_lower extremity', 'localization_neck', 'localization_scalp', 'localization_trunk', 'localization_unknown', 'localization_upper extremity']


## 5. Model Development

In [ ]:
# define augmentation for train set
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(360),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.1), ratio=(0.3, 3.3)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# define augmentation for validation set and test set
val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# define custom dataset wrapper
class HAM10000Dataset(Dataset):
    def __init__(self, dataframe, metadata_cols, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.metadata_cols = metadata_cols
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['image_path']
        image = Image.open(img_path).convert('RGB')
        label = self.df.iloc[idx]['label_idx']
        label = torch.tensor(label, dtype=torch.long)
        # metadata as tensor
        meta = self.df.iloc[idx][self.metadata_cols].values.astype(np.float32)
        meta = torch.tensor(meta, dtype=torch.float32)

        if self.transform:
            image = self.transform(image)
        return image, meta, label

# WeightedRandomSampler: handle imbalanced class
class_counts = df_train['label_idx'].value_counts().sort_index().values
class_weights = 1. / torch.tensor(class_counts, dtype=torch.float)
sample_weights = [class_weights[label] for label in df_train['label_idx']]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
    generator=torch.Generator()
)

In [ ]:
# initialize data loaders
train_dataset = HAM10000Dataset(df_train, metadata_features, transform=train_transform)
val_dataset = HAM10000Dataset(df_val, metadata_features, transform=val_test_transform)
test_dataset = HAM10000Dataset(df_test, metadata_features, transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=64, sampler=sampler, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

In [ ]:
# simulation: calculate epochs needed in training so that at lest every image seen once
def run_coverage_simulation(df, trials=1000, max_eps=50):
    counts = df['label_idx'].value_counts().sort_index().values
    class_weights = 1. / torch.tensor(counts, dtype=torch.float)
    sample_weights = class_weights[df['label_idx'].values]
    probs = sample_weights / sample_weights.sum()   # normalisasi yang benar
    n_samples = len(probs)
    results = []
    for _ in range(trials):
        seen = set()
        for epoch in range(1, max_eps + 1):
            drawn_indices = torch.multinomial(probs, n_samples, replacement=True).tolist()
            seen.update(drawn_indices)
            if len(seen) == n_samples:
                results.append(epoch)
                break
    return results

# simulate and visualize
needed_epochs = run_coverage_simulation(df_train)
plt.figure(figsize=(8, 6))
sns.kdeplot(needed_epochs, fill=True, color="teal")
plt.axvline(np.mean(needed_epochs), color='red', linestyle='--', label=f'Average: {np.mean(needed_epochs):.1f} Epochs')
plt.title("Epochs Required to Cover Entire Training Set (Weighted Sampling)")
plt.legend()
plt.show()

In [ ]:
class DoubleConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        x = F.leaky_relu(self.bn1(self.conv1(x)))
        x = F.leaky_relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        return x

class SkinCancerCNN(nn.Module):
    def __init__(self, n_metadata):
        super().__init__()
        # feature extraction
        self.block1 = DoubleConvBlock(3, 32)
        self.block2 = DoubleConvBlock(32, 64)
        self.block3 = DoubleConvBlock(64, 128)
        self.block4 = DoubleConvBlock(128, 256)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))

        self.flatten_size = 256 * 4 * 4

        self.fc1 = nn.Linear(self.flatten_size, 128)
        self.bn3 = nn.BatchNorm1d(128)
        self.dropout = nn.Dropout(0.5)

        self.fc2 = nn.Linear(128 + n_metadata, 7)  # n_metadata is dynamic

    def forward(self, x, meta):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        x = self.adaptive_pool(x)
        x = x.view(x.size(0), -1)  # flatten

        x = F.leaky_relu(self.bn3(self.fc1(x)))
        x = self.dropout(x)

        combined = torch.cat((x, meta), dim=1)
        x = self.fc2(combined)
        return x

model = SkinCancerCNN(n_metadata).to(device)
print(model)

In [ ]:
# define checkpoint
class MultiMetricCheckpoint:
    def __init__(self, patience=7, path_prefix='ham10000_focalLoss'):
        self.patience = patience
        self.counter = 0
        self.best_loss = np.Inf
        self.best_bal_acc = 0.0
        self.best_f1 = 0.0
        self.early_stop = False
        self.path_prefix = path_prefix
        
    def __call__(self, val_loss, val_labels, val_preds, model):
        # another primary metrics: well suited for imbalanced class
        bal_acc = balanced_accuracy_score(val_labels, val_preds)
        f1 = f1_score(val_labels, val_preds, average='macro')
        # early stopping logic
        if val_loss < self.best_loss:
            print(f"Loss Membaik: {self.best_loss:.4f} -> {val_loss:.4f}. Simpan model_best_loss.pth")
            self.best_loss = val_loss
            torch.save(model.state_dict(), f'{self.path_prefix}_best_loss.pth')
            self.counter = 0 # reset
        else:
            self.counter += 1
            print(f"Loss tidak membaik. EarlyStop Counter: {self.counter}/{self.patience}")
        # checkpoint: monitor on balanced accuracy
        if bal_acc > self.best_bal_acc:
            print(f"Bal. Acc Membaik: {self.best_bal_acc:.4f} -> {bal_acc:.4f}. Simpan model_best_bal_acc.pth")
            self.best_bal_acc = bal_acc
            torch.save(model.state_dict(), f'{self.path_prefix}_best_bal_acc.pth')
        # checkpoint: monitor on F1-Score
        if f1 > self.best_f1:
            print(f"F1-Score Membaik: {self.best_f1:.4f} -> {f1:.4f}. Simpan model_best_f1.pth")
            self.best_f1 = f1
            torch.save(model.state_dict(), f'{self.path_prefix}_best_f1.pth')
        if self.counter >= self.patience:
            self.early_stop = True 
        return bal_acc, f1

In [ ]:
# training setup
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
scaler = torch.amp.GradScaler(device='cuda') if torch.cuda.is_available() else None
checkpoint_handler = MultiMetricCheckpoint(patience=18, path_prefix='ham10000_CNN')
epochs = int(np.mean(needed_epochs)) + 5

In [ ]:
# training loop
history = {
    'train_loss': [],
    'val_loss': [],
    'bal_acc': [],
    'f1': []
}

for epoch in range(epochs):
    # training phase
    model.train()
    running_loss = 0.0
    for images, meta, labels in train_loader:
        images, meta, labels = images.to(device), meta.to(device), labels.to(device)

        optimizer.zero_grad()
        if scaler:
            with torch.amp.autocast(device_type='cuda'):
                outputs = model(images, meta)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images, meta)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_train_loss = running_loss / len(train_dataset)
    history['train_loss'].append(epoch_train_loss)

    # validation phase
    model.eval()
    val_loss = 0.0
    all_labels = []
    all_preds = []
    with torch.no_grad():
        for images, meta, labels in val_loader:
            images, meta, labels = images.to(device), meta.to(device), labels.to(device)
            outputs = model(images, meta)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

            preds = torch.argmax(outputs, dim=1)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    epoch_val_loss = val_loss / len(val_dataset)
    history['val_loss'].append(epoch_val_loss)

    # compute metrics & update checkpoint
    bal_acc, f1 = checkpoint_handler(epoch_val_loss, all_labels, all_preds, model)
    history['bal_acc'].append(bal_acc)
    history['f1'].append(f1)

    # step scheduler
    scheduler.step(epoch_val_loss)

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Bal Acc: {bal_acc:.4f} | F1: {f1:.4f}")

    if checkpoint_handler.early_stop:
        print("Early stopping triggered.")
        break

In [ ]:
# visualize training history
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(history['bal_acc'], label='Balanced Accuracy')
axes[1].set_title('Balanced Accuracy')
axes[1].legend()

axes[2].plot(history['f1'], label='Macro F1')
axes[2].set_title('F1 Score')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# subprogram of model evaluation: load three best models and evaluate on test set
def evaluate_model(model_path, loader, device, n_metadata):
    model = SkinCancerCNN(n_metadata).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    all_labels = []
    all_preds = []
    with torch.no_grad():
        for images, meta, labels in loader:
            images, meta = images.to(device), meta.to(device)
            outputs = model(images, meta)
            preds = torch.argmax(outputs, dim=1)
            all_labels.extend(labels.numpy())
            all_preds.extend(preds.cpu().numpy())
    return np.array(all_labels), np.array(all_preds)

In [ ]:
# define models' path
model_paths = {
    'best_loss': 'ham10000_CNN_best_loss.pth',
    'best_bal_acc': 'ham10000_CNN_best_bal_acc.pth',
    'best_f1': 'ham10000_CNN_best_f1.pth'
}

# define classes of target variable
class_names = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

# run model evaluation
for name, path in model_paths.items():
    if os.path.exists(path):
        print(f"\nEvaluating {name} model: {path}")
        y_true, y_pred = evaluate_model(path, test_loader, device, n_metadata)
        bal_acc = balanced_accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, average='macro')
        print(f"Balanced Accuracy: {bal_acc:.4f}")
        print(f"Macro F1 Score: {f1:.4f}")
        print("\nClassification Report:")
        print(classification_report(y_true, y_pred, target_names=class_names))

        # Confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
        plt.title(f'Confusion Matrix - {name}')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.show()
    else:
        print(f"model file {path} not found. Skipping.")